<a id="unit-u.97d69c527f7044d9b786d0d3b281e2f0"></a>

<div>
  <div>03-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН НАҚТЫ УАҚЫТТАҒЫ АНАЛИТИКА</div>
  <div>Файлдан, S3 және Kafka жүйелерінен деректерді жүктеу</div>
  <p>Дереккөз бен жеткізу талаптарын Stream Load, INSERT INTO SELECT арқылы S3 TVF, Routine Load және Group Commit тәсілдеріне сәйкестендіріп, содан кейін сапа сандары мен қайталау әрекетін тексеріңіз.</p>
  <span>Doris 4.1.3 · Stream Load · Label · S3 TVF · Routine Load · Kafka · Group Commit</span>
</div>

**3-модуль: Apache Doris жүйесіне деректерді жүктеу** · [Сабақ](course.md) · [Тест](quiz03_load_methods_and_retry_safety.ipynb)

Шамамен 30 минут жоспарлаңыз; алғашқы орындауда шамамен 0.2 GB Kafka образы да жүктеледі.

Бұл зертханалық жұмыс 1-зертханалық жұмыста жасалған бір түйінді, сақтау мен есептеуі біріктірілген Doris сынақ ортасында жалғасады. Оның үлкен Parquet дереккөзі — 1 және 2-зертханалық жұмыстарда қолданылған сол 10,158,080 жолдық электрондық сауда оқиғаларының жиыны. Негізгі CSV және ішкі құрылымды JSON файлдарында осы жиыннан алынған бірдей бекітілген 1,024 жазба үлгісі әр жүктеу әдісіне қажетті пішімде берілген.

Stream Load, `INSERT INTO SELECT` арқылы S3 TVF және Routine Load бөлек мақсатты кестелерге жазады. Сондықтан олардың жол сандары мен аяқталу дәлелдері дербес қалады. Kafka брокері — тағы бір жергілікті контейнер; тек Parquet нысандары мен жүктелетін файлдар қашықта орналасқан.

Деректер жиынының шығу тегі: барлық таза жазбалар [REES46](https://rees46.com/en/datasets) ұсынған ашық [бірнеше санаттағы дүкеннің электрондық сауда әрекеті деректерінен](https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/data) алынған. Бөлек он жолдық CSV ішінде `revenue` өрісінде бір жарамсыз мәтін мәні бар және ол транзакцияның қабылданбауына әкеледі.

<a id="unit-u.986d88d86f1f4021a996a74171adb002"></a>

<a id="initialize-the-lab"></a>

### Зертханалық жұмысты бастапқы баптау

Келесі ұяшықты 1-бөлімге дейін және Jupyter ядросын әр қайта іске қосқаннан кейін орындаңыз. Ол ортақ көмекші модульді қайта жүктейді, 1/2-зертханалық жұмыстардың нәтиже стильдерін орнатады және кейінгі ұяшықтар пайдаланатын `lab` нысанын жасайды. Контейнер іске қоспайды, кесте жасамайды, файл жүктемейді және дерек жүктемейді.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT);

<a id="unit-u.efd5246395864dd4980c4a6ad7060fdf"></a>

<a id="1-match-workload-requirements-to-a-load-method"></a>

## 1. Жұмыс жүктемесі талаптарын жүктеу әдісіне сәйкестендіру

Тек дереккөз түрі жүктеу әдісін таңдауға жеткіліксіз. Төмендегі әр жұмыс жүктемесі алты талапты көрсетеді: дереккөз, көлем, кідіріс, жеткізу бағыты, түрлендіру және қайталау күйі.

Әр қолжетімді жүктеу әдісін сәйкес жұмыс жүктемесіне сүйреп апарыңыз. Сүйреу қолайсыз болса, әдісті таңдап, содан кейін жүктеме карточкасын басыңыз. Қате сәйкестіктер бекітілмейді және қай талап таңдауға қайшы келетінін түсіндіреді.

In [ ]:
lab.load_method_quiz();

<a id="unit-u.432b46328c59461fb26ef48fe24f3689"></a>

<a id="2-isolate-the-three-load-methods"></a>

## 2. Үш жүктеу әдісін оқшаулау

Бұл бөлім сақталған Doris сынақ ортасына қайта қосылып, тек 3-зертханалық жұмысқа тиесілі үш мақсатты кестені қайта жасайды. 1/2-зертханалық жұмыстардың үлкен `events`, `events_v2` және `events_v3` кестелері өзгермейді.

Үш мақсат та бірдей оқиға схемасын және бір Bucket пайдаланады. Бұл шағын жергілікті кестелерге бір Bucket жеткілікті; өндірістік өлшем дерек көлеміне, BE қуатына және сұрау параллельдігіне байланысты. Бұл қадам тек кесте құрылымдарын жасайды; дереккөз оқымайды және жол жүктемейді. Әр жүктеу әдісінің бөлімі орындалғанда өзінің транзакция немесе тапсырма идентификаторын жасайды.

In [ ]:
lab.start_container()
lab.connect()
lab.execute("USE doris_course")


for table_name in (
    "events_stream", "events_s3", "events_routine"
):
    lab.execute(f"DROP TABLE IF EXISTS {table_name}")

lab.execute("""
CREATE TABLE events_stream (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    user_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL,
    product_id BIGINT NOT NULL,
    revenue DECIMAL(12, 2) NOT NULL DEFAULT "0.00"
)
DUPLICATE KEY(event_time, event_id, user_id)
DISTRIBUTED BY RANDOM BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")

lab.execute("CREATE TABLE events_s3 LIKE events_stream")
lab.execute("CREATE TABLE events_routine LIKE events_stream")

lab.sql("""
SELECT 'events_stream' AS target_table, COUNT(*) AS row_count FROM events_stream
UNION ALL
SELECT 'events_s3', COUNT(*) FROM events_s3
UNION ALL
SELECT 'events_routine', COUNT(*) FROM events_routine
ORDER BY target_table
""", title="Empty Lab 3 targets");

<a id="unit-u.19eab55dbcfc426381d4b2d333383b1b"></a>

**Күтілетін нәтиже:** сұрау үш жол қайтарады: `events_stream`, `events_s3` және `events_routine` үшін бір-бірден; әр `row_count` **0**. Әлі CSV, Parquet немесе Kafka деректері оқылған да, импортталған да жоқ.

<a id="unit-u.2b0487257bfa4207b426ed4c591ec2ac"></a>

<a id="3-load-a-local-csv-with-stream-load"></a>

## 3. Stream Load арқылы жергілікті CSV жүктеу

Stream Load клиенттегі жергілікті файлдан байттар жібереді. Келесі ұяшық тек осы жолға қажетті CSV алады және екі орнын да көрсетеді: курс bucket-індегі ашық HTTPS мекенжайын және кейін `curl -T` пәрменіне берілетін абсолют жергілікті жолды.

Жарамды жергілікті көшірме бар болса, қайта пайдаланылады. Әйтпесе жүктеу `.part` файлына жазылып, тексеріледі және атомарлы түрде қайта аталады, сондықтан үзілген тасымал жарамды сынақ файлын алмастыра алмайды.

Файлда 1/2-зертханалық жұмыс жиынынан алынған 1,024 нақты жазба бар. Баған бөлгіші ретінде `|` пайдаланылады. Doris Stream Load әдепкі `column_separator` мәнін табуляция деп алады, сондықтан бұл сұрау `column_separator:|` мәнін айқын орнатуы тиіс; сол тақырыпсыз Doris әр жолды қате өріс шекарасымен талдар еді.

Ноутбук оқуға ыңғайлы қалуы үшін алдын ала қарау тек алғашқы он жазбаны көрсетеді. Толық файлды `curl` жібереді.

In [ ]:
STREAM_CSV_PATH = "level1/module03-loading-data/downloads/stream_events.csv"

lab.fetch_fixtures([
    (
        "Stream Load CSV",
        "https://doris-courses-public-dataset.s3.us-east-1.amazonaws.com/doris-course/lab3/v2/stream_events.csv",
        STREAM_CSV_PATH,
        "stream_csv",
    ),
])
lab.preview_fixture(STREAM_CSV_PATH, "stream_csv");

<a id="unit-u.b74806714dd746cbb8e3429d91d6589e"></a>

**Күтілетін нәтиже:** дереккөз кестесі қашықтағы HTTPS URL мекенжайын, оқушының абсолют жергілікті жолын, **1,024** жолды және `downloaded` немесе `reused` күйін көрсетеді. Алдын ала қарау 1 және 2-зертханалық жұмыстарда қолданылған сол оқиғалар жиынының тік сызықпен бөлінген алғашқы он жазбасын көрсетеді.

<a id="unit-u.f7ec6b5ff4d345bd891005a964ec622e"></a>

Сұраудан бұрын мақсатты кесте тазартылады, сондықтан бұл негізгі жүктеу ұяшығын қайта орындау қайталанатын жолдарды қосудың орнына нақты жүктеу жасайды. Әр орындауды жаңа label анықтайды. Сұрау Stream Load шарттарын толық көрсетеді:

- `label` осы топтаманы анықтап, сол топтаманы қайталауды қорғайды.
- `column_separator:|` қажет, өйткені бұл CSV өрістерін тік сызықпен бөледі, ал Stream Load әдепкіде табуляция күтеді.
- `columns` алдымен мәтін мәндеріне `raw_event_time` және `raw_revenue` атауларын береді, содан кейін оларды мақсатты бағандарға сәйкестендіреді.
- `STR_TO_DATE` CSV уақыт мәтінін `DATETIME` типіне, ал `CAST` түсім мәтінін `DECIMAL(12,2)` типіне түрлендіреді.
- `strict_mode:true` жарамсыз түрлендірудің байқатпай қабылдануына жол бермейді.

In [ ]:
lab.execute("TRUNCATE TABLE events_stream")
STREAM_LABEL = f"module3_csv_{lab.new_run_suffix()}"

STREAM_REQUEST = f"""
curl --silent --show-error --location-trusted -u root: \
  -H "Expect:100-continue" \
  -H "label:{STREAM_LABEL}" \
  -H "format:csv" \
  -H "column_separator:|" \
  -H "strict_mode:true" \
  -H "max_filter_ratio:0" \
  -H "columns:raw_event_time,event_id,user_id,event_type,region,product_id,raw_revenue,event_time=str_to_date(raw_event_time,'%Y-%m-%d %H:%i:%s'),revenue=cast(raw_revenue as decimal(12,2))" \
  -T "{STREAM_CSV_PATH}" \
  -X PUT {lab.fe_http_url}/api/doris_course/events_stream/_stream_load
"""

stream_response = lab.stream_load(
    STREAM_REQUEST,
    title="Stream Load result",
    expected_status="Success",
    expected_counts={
        "NumberTotalRows": 1024,
        "NumberLoadedRows": 1024,
        "NumberFilteredRows": 0,
        "NumberUnselectedRows": 0,
    },
)

lab.verify_lab3_sample("""
SELECT
    COUNT(*) AS loaded_rows,
    COUNT(DISTINCT event_id) AS distinct_events,
    MIN(event_time) AS min_event_time,
    MAX(event_time) AS max_event_time,
    SUM(revenue) AS total_revenue
FROM events_stream
""", title="Committed Stream Load sample");

<a id="unit-u.2e4ce62aabf84ec89875ee2fb30a20cc"></a>

**Күтілетін нәтиже**

Жауап `1,024 total = 1,024 loaded + 0 filtered + 0 unselected` деп хабарлайды. Тексеру сұрауы мынаны қайтарады:

| loaded_rows | distinct_events | min_event_time | max_event_time | total_revenue |
|---:|---:|---|---|---:|
| 1024 | 1024 | 2019-12-01 03:08:44 | 2020-03-06 03:10:18 | 23299.84 |

Жасыл белгі осы бес мәннің `datasets/expected_results.yaml` ішінде тіркелген үлгіге сәйкес келетінін растайды. Файлдың соңғы жолы — 2020-03-03 оқиғасы, бірақ файлдағы ең кейінгі оқиға 2020-03-06 күнінен; Routine Load бөлімі сол бес мәнді тексереді. `Status=Success` топтама транзакциясының бекітілгенін білдіреді. Нәтиже сұраудың өткен уақытын да көрсетеді; `TxnId` пен уақыттар өзгереді. Толық жауап, Doris берген жағдайда `ErrorURL` қоса, жиналған JSON нәтижесінде қолжетімді болып қалады.

<a id="unit-u.2385b16cf8214b84add77093eb491d04"></a>

<a id="retry-the-uncertain-request-with-the-same-label"></a>

### Нәтижесі белгісіз сұрауды сол label мәнімен қайталау

Клиент Doris бекіткеннен кейін жауапты жоғалтуы мүмкін. Сол байттарды сол label мәнімен қайта жіберу сол топтама туралы сұрайды; ол бұл Duplicate Key моделі кестесінде тағы бір транзакция жасамауы немесе тағы 1,024 жол қоспауы тиіс.

Бұл қайталау үшін маңызды жауап өрістері — `status` және `label`. Doris басқа жүктеуді бастамас бұрын қайталанатын label мәнін қабылдамайды, сондықтан жаңа транзакция немесе жол сапасы нәтижесі жоқ. Төмендегі кесте саны ешбір жол қосылмағанының шешуші дәлелін береді.

In [ ]:
retry_response = lab.stream_load(
    STREAM_REQUEST,
    title="Same-label retry",
    expected_status="Label Already Exists",
    display_columns=["status", "label"],
)

lab.sql(
    "SELECT COUNT(*) AS rows_after_retry FROM events_stream",
    title="Rows after same-label retry",
);

<a id="unit-u.90e5b1fea8f9407a89c7b7b851a2a7ba"></a>

**Күтілетін нәтиже**

`Same-label retry` ішінде қайталаудан қорғауға қатысты дәлел ғана бар:

| status | label |
|---|---|
| Label Already Exists | Сәтті сұрау пайдаланған сол `module3_csv_...` белгісі |

`Rows after same-label retry` мынаны қайтарады:

| rows_after_retry |
|---:|
| 1,024 |

Кесте санының өзгермеуі Doris екінші транзакцияны жібермегенін дәлелдейді. Басқа label жаңа транзакцияны анықтап, осы Duplicate Key моделі кестесіне жолдар қосар еді.

<a id="unit-u.f7d00c276cf84281b72a971d71447657"></a>

<a id="reject-a-batch-that-exceeds-its-quality-threshold"></a>

### Сапа шегінен асатын топтаманы қабылдамау

Бөлек он жолдық CSV 1/2-зертханалық жұмыс деректер жиынынан көшірілген, тек соңғы `revenue` мәні `not-a-number` деп ауыстырылған. Stream Load жол деңгейіндегі талдау қатесін хабарлап, ішінара қабылданған топтаманы бекітпейді.

Сұрау жаңа label және `max_filter_ratio:0` пайдаланады. Сондықтан бір түрлендіру қатесі рұқсат етілген қате үлесінен асып, бүкіл транзакцияның қабылданбауына әкеледі. Doris Stream Load JSON жауабында `ErrorURL` қайтарады. Код сол өрісті алып, оған дереу HTTP GET сұрауын жібереді және жауапты арнайы кестеге айналдырмай көрсетеді. Қате журналдары уақытша диагностикалық ресурстар, сондықтан оларды сәтсіз сұраудан кейін бірден тексеріңіз.

In [ ]:
MALFORMED_CSV_PATH = "level1/module03-loading-data/downloads/stream_events_malformed.csv"

lab.fetch_fixtures([
    (
        "Malformed Stream Load CSV",
        "https://doris-courses-public-dataset.s3.us-east-1.amazonaws.com/doris-course/lab3/v2/stream_events_malformed.csv",
        MALFORMED_CSV_PATH,
        "stream_malformed_csv",
    ),
])

REJECT_LABEL = f"module3_csv_reject_{lab.new_run_suffix()}"

REJECT_REQUEST = f"""
curl --silent --show-error --location-trusted -u root: \
  -H "Expect:100-continue" \
  -H "label:{REJECT_LABEL}" \
  -H "format:csv" \
  -H "column_separator:|" \
  -H "strict_mode:true" \
  -H "max_filter_ratio:0" \
  -H "columns:raw_event_time,event_id,user_id,event_type,region,product_id,raw_revenue,event_time=str_to_date(raw_event_time,'%Y-%m-%d %H:%i:%s'),revenue=cast(raw_revenue as decimal(12,2))" \
  -T "{MALFORMED_CSV_PATH}" \
  -X PUT {lab.fe_http_url}/api/doris_course/events_stream/_stream_load
"""

reject_response = lab.stream_load(
    REJECT_REQUEST,
    title="Rejected Stream Load batch",
    expected_status="Fail",
    expected_counts={"NumberTotalRows": 10, "NumberFilteredRows": 1},
    show_response=False,
    show_json=False,
)
error_url = reject_response["ErrorURL"]
lab.show_stream_load_error_log(error_url);

<a id="unit-u.ef43f95d4b9048e2a28064d53101a1fb"></a>

**Күтілетін нәтиже**

- **URL алу:** `reject_response` — Stream Load қайтарған JSON жауабы. `reject_response["ErrorURL"]` оның `ErrorURL` өрісін оқып, Doris жасаған түпнұсқа URL көрсетеді.
- **Диагностиканы алу:** ноутбук сол URL мекенжайына дереу HTTP GET жіберіп, түпнұсқа жауапты `Content returned by ErrorURL` астында көрсетеді. Уақытша журнал әлі қолжетімді кезде қолмен орындалатын баламасы — `curl "<ErrorURL>"`.
- **Қабылданбаған жазбаны табу:** `not-a-number` қамтитын бастапқы жазбаны табыңыз. Оның тік сызықпен бөлінген өрістерінің реті `event_time | event_id | user_id | event_type | region | product_id | revenue`, сондықтан жарамсыз мән `revenue` өрісіне тиесілі.
- **Себебін оқу:** Doris 4.1.3 нұсқасында өңделмеген диагностика `column(revenue) values is null while columns is not nullable` деп көрсетуі мүмкін. `not-a-number` мәнін `DECIMAL(12,2)` типіне сәйкестендіретін `CAST` нәтижесі `NULL` болады; мақсатты `revenue` бағаны `NOT NULL`, сондықтан Doris сол жолды сүзеді. Хабар бастапқы файлда өрістің жоқтығын емес, түрлендірілген мәнді сипаттайды.
- **Транзакция нәтижесін түсіну:** жауап `Status=Fail`, `NumberTotalRows=10`, `NumberLoadedRows=0` және `NumberFilteredRows=1` деп тексеріледі. `max_filter_ratio` мәні `0` болғандықтан, тұтас он жолдық транзакция қабылданбайды және бұл сұраудың ешбір жолы бекітілмейді.

<a id="unit-u.08791643c39e4b74a9ddf186056fef47"></a>

<a id="4-load-parquet-with-s3-tvf-and-insert-into-select"></a>

## 4. Parquet файлдарын S3 TVF және INSERT INTO SELECT арқылы жүктеу

S3 TVF 1-зертханалық жұмыста қолданылған сол 40 қашықтағы Parquet нысанын оқып, оларды уақытша қатынас ретінде көрсетеді. TVF сұрауы жолдарды тұрақты сақтамайды. Жүктеудің дәл алдында `TRUNCATE TABLE` алдыңғы жолдарды жояды. Содан кейін `INSERT INTO SELECT` `events_s3` кестесіне тұрақты, синхронды көшіруді орындап, нақты енгізу уақытын хабарлайды.

Қашықтағы нысандар үлгісі: `s3://doris-courses-public-dataset/doris-course/events/v1/events-*.parquet`

SQL толық S3 TVF анықтамасын тікелей береді: нысан URI мекенжайын, endpoint, аймақ, тіркелгі режимін, екі желілік күту шегін және Parquet пішімін. Курс bucket-і ашық, сондықтан `"s3.credentials_provider_type" = "ANONYMOUS"` жабық bucket қажет ететін қатынау кілті мен құпия кілтті алмастырады. Күту шектері 1-зертханалық жұмыстағыдай: Doris әдепкі 1 және 3 секундының орнына қосылымды ашуға 10 секунд және дерексіз күтуге 30 секунд, сондықтан баяу желі немесе прокси жүктеуді сәтсіз аяқтаудың орнына баяулатады. Орындау кезінде ештеңе жасырылмайды және алмастырылмайды; көріп тұрған операторыңыз — Doris орындайтын оператор.

In [ ]:
lab.sql("""
SELECT event_time, event_id, user_id, event_type, region, product_id, revenue
FROM S3(
    "uri" = "s3://doris-courses-public-dataset/doris-course/events/v1/events-*.parquet",
    "s3.endpoint" = "https://s3.us-east-1.amazonaws.com",
    "s3.region" = "us-east-1",
    "s3.credentials_provider_type" = "ANONYMOUS",
    "s3.connection.timeout" = "10000",
    "s3.connection.request.timeout" = "30000",
    "format" = "parquet"
)
LIMIT 5
""", title="Remote Parquet preview")

lab.sql(
    "SELECT COUNT(*) AS rows_before_insert FROM events_s3",
    title="Internal target before INSERT",
);

<a id="unit-u.8ea7b38e0eaa49b39e4bfb7bd66a8724"></a>

**Күтілетін нәтиже:** қашықтағы бес оқиға жолы көрінеді, ал `rows_before_insert` **0** болып қалады. Бұл TVF сұрауы мен ішкі кестеде дерек сақтау бөлек амалдар екенін дәлелдейді.

In [ ]:
lab.execute("TRUNCATE TABLE events_s3")

lab.insert("""
INSERT INTO events_s3 (
    event_time, event_id, user_id, event_type, region, product_id, revenue
)
SELECT
    event_time, event_id, user_id, event_type, region, product_id, revenue
FROM S3(
    "uri" = "s3://doris-courses-public-dataset/doris-course/events/v1/events-*.parquet",
    "s3.endpoint" = "https://s3.us-east-1.amazonaws.com",
    "s3.region" = "us-east-1",
    "s3.credentials_provider_type" = "ANONYMOUS",
    "s3.connection.timeout" = "10000",
    "s3.connection.request.timeout" = "30000",
    "format" = "parquet"
)
""", title="S3 Parquet insert")

lab.sql("""
SELECT
    COUNT(*) AS target_rows,
    MIN(event_time) AS min_event_time,
    MAX(event_time) AS max_event_time,
    SUM(revenue) AS total_revenue
FROM events_s3
""", title="S3 batch validation");

<a id="unit-u.6c22441f4c04419eabb991df36a8b222"></a>

**Күтілетін нәтиже:** енгізу **10,158,080 әсер етілген жолды** және өткен уақытын хабарлайды. Содан кейін `events_s3` ішінде `2019-12-01 00:00:12` пен `2020-03-11 04:53:08` арасындағы **10,158,080** жол және **39,984,455.64** жалпы түсім болады. Енгізу `curlCode: 28, Timeout was reached` қатесімен тоқтаса, AWS `us-east-1` аймағына желі тіпті жоғарыдағы күту шектері үшін де тым баяу; оны 1-зертханалық жұмыста сипатталғандай тексеріп, ұяшықты қайта орындаңыз.

Жоғарыдағы Stream Load сұрауларынан айырмашылығы, бұл оператордың label мәні жоқ; 3-модульде түсіндірілгендей, `INSERT INTO events_s3 WITH LABEL <label> SELECT ...` оған label берер еді. Әр орындаудың алдында мақсатты кесте айқын тазартылады, сондықтан ноутбук толық алмастыру жасап, нақты енгізу уақытын көрсетеді. Өндірістік жұмыс барысы оның орнына қолжетімділік талаптарына сай оркестратор мен идемпотентті құрылым қолдануы тиіс.

<a id="unit-u.a022e639c39a40ecb73a2ebf8783a7ba"></a>

<a id="5-continuously-load-kafka-json-with-routine-load"></a>

## 5. Routine Load арқылы Kafka JSON хабарларын үздіксіз жүктеу

Дереккөз шектелген файл емес, хабарлар келіп тұратын Kafka тақырыбы болғанда Routine Load қолайлы. Routine Load — Doris басқаратын ұзақ жұмыс істейтін Kafka тұтынушы тапсырмасы. Doris оның микротоптама тапсырмаларын жоспарлайды, деректері мен Kafka ығысуларын бірге бекітеді және тапсырма күйін сақтайды.

Қоса берілген Compose файлы бар `doris-rta` желісінде KRaft режиміндегі бір Apache Kafka 3.9.1 брокерін іске қосады. Docker сәйкес `linux/arm64` немесе `linux/amd64` образын автоматты таңдайды. Мекенжайлар әртүрлі шақырушыларға арналған:

| Шақырушы | Брокер мекенжайы |
|---|---|
| Docker желісінің ішіндегі Doris BE | `kafka:29092` |
| Хостта жұмыс істейтін клиент | `localhost:19092` (немесе `DORIS_RTA_KAFKA_PORT` ішіндегі порт) |

JSON Lines сынақ файлы осы бөлімде алынады, өйткені Kafka консоль өндірушісі — оның алғашқы тұтынушысы. Онда Stream Load қолданған сол 1,024 логикалық жазба ішкі құрылымды JSON түрінде берілген. Нәтиже курс bucket-індегі ашық HTTPS мекенжайын және кейін shell кірісін қайта бағыттауға берілетін жергілікті жолды көрсетеді.

2-бөлімде `events_routine` бір рет жасалғаннан кейін бұл ішкі бөлімді дербес қайталауға болады. Оның үш код ұяшығын ретімен орындаңыз:

```text
1. Prepare the Kafka source
   ├── create a new Kafka topic
   ├── choose the Routine Load job name
   └── do not create the Doris job yet

2. Create the Routine Load job and mapping
   ├── define how nested JSON fields map to table columns
   ├── execute CREATE ROUTINE LOAD for the chosen topic
   └── wait until the Doris job is RUNNING

3. Publish and observe
   ├── publish JSON messages to the Kafka topic
   └── observe Doris consume and commit each batch
```

Екінші ұяшық жариялаудан бұрын міндетті түрде орындалуы тиіс: бірінші ұяшық Kafka тақырыбын жасап, тапсырма атауын белгілейді, бірақ Routine Load тапсырмасын тек `CREATE ROUTINE LOAD` жасайды. Kafka дайындау ұяшығын қайта орындау қажет болса оның алдыңғы тапсырмасын тоқтатып, жаңа тақырып, тұтынушы тобы және тапсырма атауын таңдайды. Kafka дайындау сәтті аяқталғаннан кейін ғана ұсынылған атаулар белсенді ноутбук айнымалыларына айналады, сондықтан үзілген дайындау кейінгі ұяшықтарды жасалмаған тақырыпқа бағыттап қалдыра алмайды. Routine Load тапсырмасының ұяшығы жаңа тапсырманы жасамас бұрын `events_routine` кестесін тазартады. Сондықтан Kafka хабарлары мен бекітілген ығысулар алдыңғы орындаудан жаңасына өте алмайды.

In [ ]:
lab.connect()
lab.execute("USE doris_course")

previous_routine_job = globals().get("ROUTINE_JOB")

next_routine_run_suffix = lab.new_run_suffix()
next_kafka_topic = f"doris-events-json-{next_routine_run_suffix.replace('_', '-')}"
next_routine_job = f"module3_events_json_{next_routine_run_suffix}"

lab.stop_routine_if_exists(previous_routine_job)

ROUTINE_JSON_PATH = "level1/module03-loading-data/downloads/routine_events.jsonl"

lab.fetch_fixtures([
    (
        "Routine Load JSON Lines",
        "https://doris-courses-public-dataset.s3.us-east-1.amazonaws.com/doris-course/lab3/v2/routine_events.jsonl",
        ROUTINE_JSON_PATH,
        "routine_json",
    ),
])

lab.shell(fr"""
set -euo pipefail

echo "[1/3] Start the pinned single-broker Kafka sandbox"
KAFKA_PORT="${{DORIS_RTA_KAFKA_PORT:-19092}}"
if [ "$(docker inspect --format '{{{{.State.Running}}}}' doris-rta-kafka 2>/dev/null || true)" != "true" ] \
   && (exec 3<>"/dev/tcp/127.0.0.1/${{KAFKA_PORT}}") 2>/dev/null; then
  echo "Host port ${{KAFKA_PORT}} is already in use by another program:"
  lsof -nP -iTCP:"${{KAFKA_PORT}}" -sTCP:LISTEN || true
  echo "Stop that program, or set a free port before starting Jupyter, for example:"
  echo "  export DORIS_RTA_KAFKA_PORT=29092"
  exit 9
fi
if ! docker compose -f level1/module03-loading-data/compose.kafka.yml up -d; then
  echo "Kafka did not start. If the image could not be downloaded, check access to Docker Hub"
  echo "(proxy or VPN), then try: docker pull apache/kafka:3.9.1"
  exit 10
fi

echo "[2/3] Wait for the Kafka health check"
for attempt in $(seq 1 60); do
  STATUS=$(docker inspect --format '{{{{if .State.Health}}}}{{{{.State.Health.Status}}}}{{{{else}}}}none{{{{end}}}}' doris-rta-kafka)
  [ "$STATUS" = "healthy" ] && break
  [ "$attempt" -eq 60 ] && {{ docker logs --tail 100 doris-rta-kafka; exit 1; }}
  sleep 2
done

echo "[3/3] Create an isolated topic for this Lab run"
docker exec doris-rta-kafka /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server kafka:29092 \
  --create --if-not-exists \
  --topic "{next_kafka_topic}" \
  --partitions 1 \
  --replication-factor 1

docker compose -f level1/module03-loading-data/compose.kafka.yml ps
""", title="Prepare the Kafka source")

lab.preview_fixture(ROUTINE_JSON_PATH, "routine_json")

ROUTINE_RUN_SUFFIX = next_routine_run_suffix
KAFKA_TOPIC = next_kafka_topic
ROUTINE_JOB = next_routine_job;

<a id="unit-u.3d6ebb315b214ceabb6394bd5b26fa41"></a>

**Күтілетін нәтиже:** Kafka `healthy` күйіне жетеді, бірегей тақырып бір бөліммен жасалады және дереккөз кестесі **1,024** JSON хабарын көрсетеді. Алдын ала қарау Stream Load қолданған сол таза жазбалардың алғашқы он ішкі құрылымды көрінісін көрсетеді. Бұл сәтте Kafka дереккөзі бар, бірақ Doris Routine Load тапсырмасы жоқ; хабарларды жарияламас бұрын келесі ұяшықты орындаңыз.

Kafka томы тұрақты сақталады, бірақ бірегей тақырып ескі зертханалық жұмыс хабарларының осы орындауға енуіне жол бермейді. Төмендегі өндіруші ұяшығы ішінара толған, күйі екіұшты тақырыпты да қабылдамайды.

**Жиі кездесетін мәселелер**

- **Ұяшық 1-қадамда "Host port 19092 is already in use" деп тоқтайды.** Басқа бағдарлама, көбіне басқа Kafka, сол портты тыңдап тұр. Ұяшық оны көрсетеді. Оны тоқтатыңыз немесе Jupyter бастамас бұрын `DORIS_RTA_KAFKA_PORT` мәнін бос портқа орнатыңыз; Doris Docker желісінің ішіндегі Kafka-ға өзгермейтін `kafka:29092` мекенжайымен жетеді.
- **Ұяшық "Kafka did not start" деп тоқтайды.** Жоғарыдағы Docker нәтижесін оқыңыз. `apache/kafka:3.9.1` образы жүктелмесе, прокси немесе VPN арқылы Docker Hub қолжетімділігін тексеріп, терминалда `docker pull apache/kafka:3.9.1` орындаңыз.
- **Күйді тексеру ешқашан өтпейді.** Ұяшық Kafka журналының соңғы 100 жолын шығарады. `docker logs --tail 100 doris-rta-kafka` да соны көрсетеді.

<a id="unit-u.1d1c6e0e05a34264a744f92c1ec79dbe"></a>

<a id="create-the-routine-load-job-and-define-its-json-mapping"></a>

### Routine Load тапсырмасын жасау және оның JSON сәйкестендіруін анықтау

Бұл ұяшық келесі жариялау ұяшығына қажетті Routine Load тапсырмасын жасайды. `CREATE ROUTINE LOAD` таңдалған тапсырма атауын Kafka тақырыбымен байланыстырады, ал `jsonpaths` әр ішкі құрылымды хабардан жеті мәнді бекітілген ретпен шығарады. `COLUMNS` сол мәндерді мақсатты бағандарға немесе уақытша атауларға сәйкестендіреді: ISO уақыт жолы `DATETIME` типіне түрленеді, ал JSON ішіндегі бас әріпті оқиға түрі 1/2-зертханалық жұмыс жиынындағы кіші әріпті мәнге қайта айналады. Сандық JSON түсімі мақсатты `DECIMAL(12,2)` типіне сәйкестенеді.

Барлық 1,024 хабар жарамды, сондықтан жарияланған хабарлар мен бекітілген мақсатты жолдар арасындағы кез келген айырмашылық жол сапасы бойынша сүзуді емес, тұтыну ілгерілеуін көрсетеді. Тапсырма жасалардың дәл алдында мақсатты кесте тазартылады, бұл кейін байқалған барлық жолдың осы орындауға тиесілі болуын қамтамасыз етеді.

In [ ]:
lab.execute("TRUNCATE TABLE events_routine")

lab.execute(fr"""
CREATE ROUTINE LOAD doris_course.{ROUTINE_JOB} ON events_routine
COLUMNS(
    raw_event_time,
    event_id,
    user_id,
    raw_event_type,
    raw_region,
    product_id,
    revenue,
    event_time = STR_TO_DATE(raw_event_time, '%Y-%m-%dT%H:%i:%s'),
    event_type = LOWER(raw_event_type),
    region = raw_region
)
PROPERTIES (
    "format" = "json",
    "jsonpaths" = "[\"$.meta.occurred_at\",\"$.meta.event_id\",\"$.actor.user_id\",\"$.event.type\",\"$.geo.region\",\"$.product_id\",\"$.event.revenue\"]",
    "desired_concurrent_number" = "1",
    "max_batch_interval" = "5",
    "strict_mode" = "true",
    "max_filter_ratio" = "0"
)
FROM KAFKA (
    "kafka_broker_list" = "kafka:29092",
    "kafka_topic" = "{KAFKA_TOPIC}",
    "property.kafka_default_offsets" = "OFFSET_BEGINNING",
    "property.group.id" = "{ROUTINE_JOB}"
)
""")

lab.wait_for_routine_state(ROUTINE_JOB, "RUNNING")
lab.routine_load_status(ROUTINE_JOB);

<a id="unit-u.ff161201b8b3458695b6fe93ff9c6d1d"></a>

**Күтілетін нәтиже:** көмекші құрал уақытша `NEED_SCHEDULE` күйінде күтіп, тапсырманы тек `RUNNING` күйіне жеткеннен кейін көрсетеді. Бұл Routine Load тапсырмасының енді бар екенін және келесі жариялау ұяшығына дайын екенін растайды. Хабарлар жарияланғанға дейін жалпы, жүктелген және қате санағыштары нөлге тең.

<a id="unit-u.4c90f3945f6e43c29d566bc85477c553"></a>

<a id="publish-messages-and-observe-committed-progress"></a>

### Хабарларды жариялау және бекітілген ілгерілеуді бақылау

Өндіруші әр JSON жолын бір Kafka хабары ретінде жібереді. Ол 1,024 хабарды 256-дан төрт топтамаға бөліп жариялайды. Әр топтамадан кейін нәтиже Kafka және Doris санағыштарын дереу тіркейді, содан кейін Routine Load сол бақылау нүктесін бекіткенше оларды секунд сайын жаңартады. Өндіруші келесі топтаманы жарияламас бұрын екі секунд күтеді.

Тікелей құбыр `Notebook producer → Kafka topic → Routine Load job → events_routine` түрінде сол орнында жаңартылады. Ол уақытша кезек жиналуын айқын көрсетеді: `waiting for Routine Load = Kafka published − Routine Load committed`. Kafka тұтынылған хабарларды сақтау мерзімі саясаты бойынша сақтайды, сондықтан бұл мән Routine Load тапсырмасы әлі бекітпеген хабарларды білдіреді; ол тұтынылған хабарлардың Kafka-дан физикалық жойылғанын білдірмейді.

Құбыр астындағы тарих әр әрекеттің орындаушысын атайды. `Notebook producer · published 256` өндірушінің алғашқы топтаманы Kafka-ға қосқанын білдіреді. `Routine Load job · waiting for 256` — сол хабарлар бекітілмей тұрғандағы бір секундтық тексеру. `Routine Load job · committed 256` тапсырманың сол бақылау нүктесін бекітіп, жолдардың `events_routine` кестесінде сұрауға қолжетімді болғанын білдіреді.

In [ ]:
lab.publish_kafka_batches(
    topic=KAFKA_TOPIC,
    fixture_path=ROUTINE_JSON_PATH,
    job_name=ROUTINE_JOB,
    target_table="events_routine",
    batch_size=256,
    poll_interval_seconds=1,
    batch_pause_seconds=2,
)

lab.verify_lab3_sample("""
SELECT
    COUNT(*) AS loaded_rows,
    COUNT(DISTINCT event_id) AS distinct_events,
    MIN(event_time) AS min_event_time,
    MAX(event_time) AS max_event_time,
    SUM(revenue) AS total_revenue
FROM events_routine
""", title="Committed Routine Load sample");

<a id="unit-u.bd1194d0c43a465c9989c92658cf51e9"></a>

**Күтілетін нәтиже**

Тікелей құбыр нөл жарияланған хабардан, нөл күтіп тұрған хабардан және нөл Doris жолынан басталады. Әр топтама үшін `Kafka published` алдымен 256-ға артады, ал `Routine Load committed` және `events_routine rows` әлі алдыңғы бақылау нүктесін көрсетуі мүмкін. Осы аралықта `waiting for Routine Load` мәні 256. Содан кейін әр топтама 256, 512, 768 және 1,024 жолда `Routine Load job · committed` жолын қосады. Соңғы бақылау нүктесінде:

- `Kafka published`, `Routine Load committed` және `events_routine rows` бәрі 1,024.
- `waiting for Routine Load` — 0, `errors` — 0 болып қалады және `Routine Load state` — `RUNNING` болып қалады.
- Тексеру сұрауы 1,024 бірегей оқиға ID мәнін, `2019-12-01 03:08:44` пен `2020-03-06 03:10:18` арасындағы уақыт ауқымын және **23,299.84** жалпы түсімді көрсетеді: бұл Stream Load үлгісіндегі сол мәндер, оларды жасыл белгі растайды.

`Routine Load job · waiting` көріністерінің нақты саны жергілікті контейнерлерге байланысты. Жылдам Routine Load тапсырмасы алғашқы бір секундтық тексеруден бұрын бекітуі мүмкін, бірақ бөлек өндіруші және Routine Load тапсырмасы жолдары әр әрекетті кім орындағанын бәрібір көрсетеді. Топтамалар арасындағы екі секундтық үзіліс оқу қарқыны үшін берілген; санағыштар мен бекітулер нақты, жасанды ілгерілеу анимациясы емес.

<a id="unit-u.5964a2b1829b4b72847683ee207beb3a"></a>

<a id="pause-and-resume-the-routine-load-job"></a>

### Routine Load тапсырмасын кідірту және жалғастыру

Кідірту тапсырманы, бекітілген ығысуларды немесе мақсатты жолдарды жоймай, жаңа Routine Load қысқа тапсырмаларын тоқтатады. Жалғастыру ұзақ тапсырманы `NEED_SCHEDULE` арқылы `RUNNING` күйіне қайтарады; бекітіліп қойған Kafka хабарлары қайта тұтынылмайды.

In [ ]:
lab.execute(f"PAUSE ROUTINE LOAD FOR {ROUTINE_JOB}")
lab.wait_for_routine_state(ROUTINE_JOB, "PAUSED")
lab.routine_load_status(ROUTINE_JOB);

<a id="unit-u.a1d180f715064299b9e915068d3f253b"></a>

**Күтілетін нәтиже:** тапсырма күйі `PAUSED`; бекітілген 1,024 жолдың бәрі сұрауға қолжетімді болып қалады.

In [ ]:
lab.execute(f"RESUME ROUTINE LOAD FOR {ROUTINE_JOB}")
lab.wait_for_routine_state(ROUTINE_JOB, "RUNNING")
lab.routine_load_status(ROUTINE_JOB);

<a id="unit-u.cb046b40305f4b3eaa1114b0b14f0ba9"></a>

**Күтілетін нәтиже:** тапсырма сол 1,024 хабар ілгерілеуімен және 1,024 мақсатты жолмен `RUNNING` күйіне оралады. Сол JSON мазмұнын қайта жариялау 1,024 жаңа Kafka хабарын жасар еді; бұл Routine Load қайталауы емес, жаңа жазу.

<a id="unit-u.9968a1167f6f48e282d541c50e586b5a"></a>

<a id="stop-the-routine-load-job-permanently"></a>

### Routine Load тапсырмасын түпкілікті тоқтату

Кідірту/жалғастыруды бақылауды аяқтағаннан кейін осы тазалау ұяшығын орындаңыз. `STOP` түпкілікті: тапсырманы жалғастыру мүмкін емес, бірақ `events_routine` кестесіне бекітілген барлық жол Doris ішінде қалады.

In [ ]:
lab.execute(f"STOP ROUTINE LOAD FOR {ROUTINE_JOB}")
lab.wait_for_routine_state(ROUTINE_JOB, "STOPPED")
lab.routine_load_status(ROUTINE_JOB);

<a id="unit-u.5c4b9fd0d96b4b859279de7ca729380a"></a>

**Күтілетін нәтиже:** тапсырма күйі `STOPPED`; 1,024 мақсатты жол сұрауға қолжетімді болып қалады.

<a id="unit-u.06f7108e57cc444c86d1f5af00733c16"></a>

<a id="6-practice-let-the-quality-threshold-accept-the-batch"></a>

## 6. Жаттығу: сапа шегіне топтаманы қабылдауға мүмкіндік беру

3-бөлім бір жарамсыз `revenue` мәні бар он жолдық файлды `max_filter_ratio:0` арқылы жіберді және Doris бүкіл топтаманы қабылдамады. Сол файлды `LIKE events_stream` арқылы жасалған жаңа `events_stream_practice` кестесіне бір өзгеріспен қайта жіберіңіз: `max_filter_ratio:0.1`.

Орындамас бұрын төрт мәнді болжаңыз: жауаптың `Status`, `NumberLoadedRows` және `NumberFilteredRows` мәндерін және одан кейінгі `events_stream_practice` ішіндегі жол санын. Содан кейін сұрауды жазыңыз; 3-бөлімнен `REJECT_REQUEST` көшіріп, кестені, label және үлесті өзгерте аласыз. Аяқтағаннан кейін жаттығу кестесін жойыңыз.

In [ ]:
# Your code here

<a id="unit-u.88ce5a7587ae45d697439831b2886d86"></a>

Келесі ұяшық — үлгі шешім; ол жиналған. Салыстыру үшін оны ашыңыз немесе орындаңыз.

In [ ]:
lab.execute("DROP TABLE IF EXISTS events_stream_practice")
lab.execute("CREATE TABLE events_stream_practice LIKE events_stream")

PRACTICE_REQUEST = f"""
curl --silent --show-error --location-trusted -u root: \
  -H "Expect:100-continue" \
  -H "label:module3_csv_practice_{lab.new_run_suffix()}" \
  -H "format:csv" \
  -H "column_separator:|" \
  -H "strict_mode:true" \
  -H "max_filter_ratio:0.1" \
  -H "columns:raw_event_time,event_id,user_id,event_type,region,product_id,raw_revenue,event_time=str_to_date(raw_event_time,'%Y-%m-%d %H:%i:%s'),revenue=cast(raw_revenue as decimal(12,2))" \
  -T "{MALFORMED_CSV_PATH}" \
  -X PUT {lab.fe_http_url}/api/doris_course/events_stream_practice/_stream_load
"""
lab.stream_load(
    PRACTICE_REQUEST,
    title="The same batch with max_filter_ratio:0.1",
    expected_status="Success",
    expected_counts={"NumberTotalRows": 10, "NumberLoadedRows": 9, "NumberFilteredRows": 1},
    show_json=False,
)
lab.sql("SELECT COUNT(*) AS committed_rows FROM events_stream_practice",
        title="Rows committed by the practice batch")
lab.execute("DROP TABLE events_stream_practice");

<a id="unit-u.cc1639b466ee46eca632039f7712d07e"></a>

**Күтілетін нәтиже**

Жауап **Status `Success`**, жалпы **10** жол, **9** жүктелген және **1** сүзілген жол деп хабарлайды, ал `committed_rows` мәні **9**. Шешімдегі тексерулер үнсіз сәтті өтеді.

`max_filter_ratio` — топтама бәрібір бекітіле отырып, сүзуге болатын жолдарының үлесі. Онның бір жарамсыз жолы — 0.1 үлесі, ол 0.1 мәнінен аспайды, сондықтан транзакция тоғыз жарамды жолды бекітіп, жарамсызын алып тастайды; `0` болғанда бәрін қабылдамауы тиіс еді. Нөлден жоғары үлес толықтық пен қолжетімділік арасындағы таңдауды білдіреді: оны дереккөз шартына қарай шешіп, алынған жолдардың `ErrorURL` диагностикасын сақтаңыз. Соңында жаттығу кестесі жойылады, сондықтан зертханалық жұмыстың өз кестелері өзгермейді.

<a id="unit-u.aa8c5cc8cae043308b2272cdc27d2a7f"></a>

<a id="stop-the-kafka-sandbox"></a>

### Kafka сынақ ортасын тоқтату

Kafka үдерісін және хосттың 19092 портын босатқыңыз келгенде осы қосымша ұяшықты орындаңыз. `kafka-data` атаулы томы және барлық Doris кестелері қолжетімді болып қалады.

In [ ]:
lab.shell(r"""
set -euo pipefail

docker compose -f level1/module03-loading-data/compose.kafka.yml stop kafka
docker inspect --format 'container={{.State.Status}}' doris-rta-kafka
""", title="Stop the Kafka sandbox");

<a id="unit-u.ce1bab2464dd492a96bf3105a5213a9c"></a>

**Күтілетін нәтиже:** Docker Kafka үшін `container=exited` деп хабарлайды.

<a id="unit-u.7d259c512fb140bb869033586c3baf98"></a>

<a id="restart-the-kafka-sandbox"></a>

### Kafka сынақ ортасын қайта іске қосу

Бар брокер мен тақырыптарды қайта пайдаланғыңыз келгенде осы ұяшықты орындаңыз. Ол бар Kafka контейнерін іске қосады немесе қайта пайдаланады және оның күй тексеруін күтеді. Түпкілікті тоқтатылған Routine Load тапсырмасы тоқтаған күйінде қалады.

Бұл қайта іске қосу ұяшығы идемпотентті: оны Docker Desktop пен контейнер тоқтаған, іске қосылып жатқан немесе жұмыс істеп тұрған кезде орындауға болады. macOS жүйесінде қажет болса Docker Desktop ашылады; Linux жүйесінде ұяшықты орындамас бұрын Docker Engine іске қосыңыз.


In [ ]:
lab.start_container("doris-rta-kafka");

<a id="unit-u.207c8837f44444d7a91e192ade93743e"></a>

**Күтілетін нәтиже:** Docker `container=running health=healthy` деп хабарлайды.

<a id="lab-complete"></a>

## Зертханалық жұмыс аяқталды

Жұмыс жүктемесі талаптары бойынша жүктеу әдістерін таңдадыңыз, нақты оқиға үлгісін Stream Load арқылы жүктедіңіз, бөлек қате файлмен сол label мәнімен қайталаудың қауіпсіздігін және транзакция деңгейіндегі қабылдамауды дәлелдедіңіз, S3 TVF тексеруін атомарлы `INSERT INTO SELECT` операторынан ажыраттыңыз және сол логикалық үлгіні ішкі құрылымы бар Kafka JSON хабарларынан Routine Load тапсырмасы арқылы жүктедіңіз.

Негізгі айырмашылықтар:

- Stream Load — синхронды JSON жауабы және label негізіндегі қайталанудан қорғауы бар шектелген клиент жіберуі.
- S3 TVF қашықтағы файлдарды уақытша қатынас ретінде көрсетеді; `INSERT INTO SELECT` атомарлы топтаманы тұрақты сақтайды.
- Routine Load — Doris басқаратын ұзақ жұмыс істейтін Kafka тұтынушы тапсырмасы; оның микротоптама транзакциялары деректер мен ығысуларды бірге бекітеді.
- Group Commit жиі шағын Stream Load немесе INSERT жазуларын оңтайландырады; ол объектілік сақтау немесе Kafka коннекторын алмастырмайды.
- Сүзілген жол және тұтас қабылданбаған транзакция — әртүрлі нәтиже: алдымен жол түрлендіруден өтпейді, содан кейін сапа шегі топтаманың бекітілетінін анықтайды.

Ресми анықтамалар: [Жүктеуге шолу](https://doris.apache.org/docs/4.x/data-operate/import/load-manual/) · [Stream Load](https://doris.apache.org/docs/4.x/data-operate/import/import-way/stream-load-manual/) · [INSERT INTO SELECT](https://doris.apache.org/docs/4.x/data-operate/import/import-way/insert-into-manual/) · [Routine Load](https://doris.apache.org/docs/4.x/data-operate/import/import-way/routine-load-manual/) · [Group Commit](https://doris.apache.org/docs/4.x/data-operate/import/load-best-practices/group-commit-manual/) · [Apache Kafka Docker образы](https://kafka.apache.org/39/getting-started/docker/)

<a id="unit-u.1841d05dcf3b4eef95464658ccf175a6"></a>

<a id="next"></a>

## Келесі қадам

[3-тест: Apache Doris жүйесіне деректерді жүктеу](quiz03_load_methods_and_retry_safety.ipynb) ашыңыз. Бұл — 1-деңгейдің соңғы зертханалық жұмысы; 2-деңгей [4-модуль: Apache Doris жүйесінде деректерді модельдеу](../../level2/module04-modeling/course.md) бөлімінен басталады.